# GRPO post-training for a pretrained AEMO Decision Transformer

This notebook shows how to run **Group Relative Policy Optimization (GRPO)** as an online post-training stage on top of a pretrained AEMO Decision Transformer.

## Prerequisites
- A pretrained AEMO DT checkpoint, for example from `models/aemo/dt/.../dt_model.pt`.
- Matching model kwargs JSON, for example `configs/aemo_decision_transformer_model_kwargs.json` or the JSON saved next to your checkpoint.
- AEMO cache access for the scenario you want to post-train on. If you want cache-only runs, export `AEMO_CACHE_ONLY=1` before starting Jupyter.
- If you need to create the checkpoint first, prefer `python3 src/launch_aemo_training.py --run-tier proxy-baseline` or `learning-baseline` from the repo root.


In [ ]:
from pathlib import Path
import os
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').exists() and (candidate / 'src').exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
for _extra_path in (REPO_ROOT, REPO_ROOT / 'src'):
    _extra_str = str(_extra_path)
    if _extra_str not in sys.path:
        sys.path.insert(0, _extra_str)

REPO_ROOT


In [ ]:
import json
from datetime import datetime

import numpy as np
import polars as pl
import torch

from aemo_notebook_utils import create_aemo_env, fetch_and_preprocess_aemo_scenarios, resolve_battery_variants
from decision import AEMOAgent
from huggingface_hub import hf_hub_download
from grpo_posttraining import GRPOPrompt, GRPOTrainer, load_pretrained_dt_for_grpo, sample_rtg_values

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


In [ ]:
CACHE_DIR = REPO_ROOT / 'data' / 'aemo'
# Model architecture (matches the HF checkpoint from MoLab training)
MODEL_KWARGS = {
    'state_dim': 18,
    'act_dim': 9,
    'n_block': 8,
    'h_dim': 384,
    'context_len': 180,
    'n_heads': 8,
    'drop_p': 0.15,
    'max_timestep': 100000,
    'return_scale': 2.0,
}
# Download pretrained DT from HuggingFace
HF_REPO = 'mrvictoru/energydecision-dt'
CHECKPOINT_PATH = hf_hub_download(repo_id=HF_REPO, filename='aemo_dt_fcas_model.pt')
GRPO_OUTPUT_PATH = REPO_ROOT / 'models' / 'aemo' / 'dt' / 'dt_model_grpo.pt'

SCENARIO = {
    'label': 'nsw1_short_window',
    'region': 'NSW1',
    'start_date': datetime(2024, 1, 1),
    'end_date': datetime(2024, 1, 14),
}

BATTERY_VARIANT = {
    'name': 'medium',
    'capacity_mwh': 10.0,
    'max_power_mw': 5.0,
    'init_soc_ratio': 0.5,
}

ACTION_MODE = 'full_fcas'  # 9-dim: energy + 8 FCAS services
DEGRADATION_MODE = 'real_world'
DEGRADATION_CHEMISTRY = 'LFP'
DEGRADATION_TEMPERATURE = 30.0
STEP_DURATION_HOURS = 5.0 / 60.0
EPISODE_HOURS = 24.0
RANDOM_EPISODE_START = True
BASE_SEED = 2026

BASELINE_EVAL_EPISODES = 3
RTG_COUNT = 4
OPTIMAL_RTG = None  # auto-set from model.return_scale
RTG_SPREAD = 3.0
RTG_COUNT = 4
RTG_DIST = 'gaussian'  # 'gaussian', 'uniform', 'lognormal'
TARGET_RTG = 0.0  # default for baseline eval
DT_GAMMA = 1.0

GRPO_ITERATIONS = 3
GRPO_GROUP_SIZE = 4
GRPO_UPDATE_EPOCHS = 2
GRPO_MINIBATCH_SIZE = 64
GRPO_LR = 1e-5
GRPO_CLIP_RATIO = 0.2
GRPO_KL_COEFF = 0.02
GRPO_ENTROPY_COEFF = 0.0
GRPO_INITIAL_LOG_STD = -1.0
GRPO_TRAINABLE_LOG_STD = True

CHECKPOINT_PATH, MODEL_CONFIG_PATH


In [ ]:
# HF download already validated the files exist
print(f'Loaded pretrained DT from: {CHECKPOINT_PATH}')

processed_by_label, scenario_manifest = fetch_and_preprocess_aemo_scenarios(
    scenarios=[SCENARIO],
    cache_dir=CACHE_DIR,
    step_duration=STEP_DURATION_HOURS,
    refresh=False,
)
scenario_label = scenario_manifest[0]['label']
processed_df = processed_by_label[scenario_label]
battery_variant = resolve_battery_variants([BATTERY_VARIANT])[0]
max_step = max(1, min(processed_df.height, int(round(EPISODE_HOURS / STEP_DURATION_HOURS))))

model_kwargs = MODEL_KWARGS

expected_act_dim = {'simple': 1, 'multi_market': 3, 'full_fcas': 9}.get(ACTION_MODE, 1)
if int(model_kwargs['act_dim']) != expected_act_dim:
    raise ValueError(f'Model act_dim={model_kwargs["act_dim"]} does not match action_mode={ACTION_MODE!r}.')
if int(model_kwargs['state_dim']) != 18:
    raise ValueError(f'Expected AEMO state_dim=18, got {model_kwargs["state_dim"]}.')

processed_df.head()


In [ ]:
model, reference_model = load_pretrained_dt_for_grpo(
    model_kwargs,
    CHECKPOINT_PATH,
    device=device,
)
model.return_scale

# Set optimal RTG from model calibration
if OPTIMAL_RTG is None:
    OPTIMAL_RTG = float(model.return_scale)
print(f"Optimal RTG = {OPTIMAL_RTG}")


In [ ]:
def make_env(random_episode_start=RANDOM_EPISODE_START):
    return create_aemo_env(
        processed_data=processed_df,
        battery_variant=battery_variant,
        max_step=max_step,
        step_duration=STEP_DURATION_HOURS,
        action_mode=ACTION_MODE,
        degradation_mode=DEGRADATION_MODE,
        degradation_chemistry=DEGRADATION_CHEMISTRY,
        degradation_temperature=DEGRADATION_TEMPERATURE,
        random_episode_start=random_episode_start,
    )

def evaluate_dt_policy(dt_model, episodes=3, rtg_value=TARGET_RTG, random_episode_start=RANDOM_EPISODE_START):
    episode_rows = []
    for seed in range(episodes):
        env = make_env(random_episode_start=random_episode_start)
        agent = AEMOAgent(
            env,
            algorithm='dt',
            model=dt_model,
            rtg_value=rtg_value,
            dt_gamma=DT_GAMMA,
            reset_seed=BASE_SEED + seed if random_episode_start else None,
        )
        episode_df, _ = agent.run_episode()
        info_series = episode_df['info'].struct.unnest()
        episode_rows.append({
            'episode': seed,
            'reward_sum': float(episode_df['reward'].sum()),
            'energy_revenue': float(info_series['energy_revenue'].sum()),
            'fcas_revenue': float(info_series['fcas_revenue'].sum()),
            'total_revenue': float(info_series['total_revenue'].tail(1).item()),
        })
    return pl.DataFrame(episode_rows)

baseline_eval = evaluate_dt_policy(model, episodes=BASELINE_EVAL_EPISODES, rtg_value=TARGET_RTG)
baseline_eval


In [ ]:
# Sample RTG values around the optimal for group diversity
# Each prompt has a different RTG; group-relative advantages
# compare different strategies against each other.
rtg_values = sample_rtg_values(
    optimum=OPTIMAL_RTG, spread=RTG_SPREAD,
    count=RTG_COUNT, distribution=RTG_DIST,
    seed=BASE_SEED,
)
print(f"RTG prompts: {[round(v, 2) for v in rtg_values]}")

prompts = [
    GRPOPrompt(
        seed=BASE_SEED + idx,
        options={
            'random_episode_start': RANDOM_EPISODE_START,
        },
        rtg_value=rtg,
        max_steps=max_step,
    )
    for idx, rtg in enumerate(rtg_values)
]

trainer = GRPOTrainer(
    model,
    reference_model=reference_model,
    device=device,
    lr=GRPO_LR,
    clip_ratio=GRPO_CLIP_RATIO,
    kl_coeff=GRPO_KL_COEFF,
    entropy_coeff=GRPO_ENTROPY_COEFF,
    initial_log_std=GRPO_INITIAL_LOG_STD,
    trainable_log_std=GRPO_TRAINABLE_LOG_STD,
)

history = trainer.train(
    make_env,
    prompts=prompts,
    iterations=GRPO_ITERATIONS,
    group_size=GRPO_GROUP_SIZE,
    update_epochs=GRPO_UPDATE_EPOCHS,
    minibatch_size=GRPO_MINIBATCH_SIZE,
    dt_gamma=DT_GAMMA,
)
pl.DataFrame(history)


In [ ]:
post_grpo_eval = evaluate_dt_policy(model, episodes=BASELINE_EVAL_EPISODES, rtg_value=TARGET_RTG)
print('Baseline evaluation:')
print(baseline_eval)
print('\nPost-GRPO evaluation:')
print(post_grpo_eval)
print('\nMean reward improvement:', float(post_grpo_eval['reward_sum'].mean() - baseline_eval['reward_sum'].mean()))
print('Mean FCAS revenue improvement:', float(post_grpo_eval['fcas_revenue'].mean() - baseline_eval['fcas_revenue'].mean()))


In [ ]:
GRPO_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'meta': {'return_scale': float(getattr(model, 'return_scale', 1.0))},
    },
    GRPO_OUTPUT_PATH,
)
print(f'Saved GRPO-updated AEMO DT weights to {GRPO_OUTPUT_PATH}')


## Notes
- `ACTION_MODE` must stay aligned with the checkpoint (`simple` => `act_dim=1`, `multi_market` => `act_dim=3`, `full_fcas` => `act_dim=9`).
- For faster smoke runs, shorten `EPISODE_HOURS`, reduce `RTG_COUNT`, or drop `GRPO_ITERATIONS` to 1.
- For broader post-training, swap in a different scenario window or build a prompt list with explicit `episode_start_idx` values.
- The prompt list uses `sample_rtg_values()` — RTGs are sampled around `OPTIMAL_RTG` with `RTG_SPREAD`. Adjust these to control diversity.
- The pretrained DT is downloaded from HuggingFace (`mrvictoru/energydecision-dt`). The checkpoint file `aemo_dt_fcas_model.pt` contains the model trained with `aw=0.999, sw=0.002, rw=0.0001` on MoLab.
